In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import xgboost as xgb

from typing import List, Dict, Tuple, Callable
import os
import gc
import traceback
import warnings
from pdb import set_trace

from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.metrics import mean_squared_error as mse

In [2]:
phishing = pd.read_csv("PhiUSIIL_Phishing_URL_Dataset.csv")

In [3]:
phishing_clean = phishing.drop(["FILENAME", 'URL', 'URLLength', 'Domain', 'Title', 'TLD'], axis = 1)

In [4]:
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

X = phishing_clean.drop('label', axis=1).values
y = phishing_clean['label'].values

print("Original class distribution:", Counter(y))

# Initialize undersampler
undersampler = RandomUnderSampler(sampling_strategy=1.0, random_state=42)

# Apply undersampling
X_resampled, y_resampled = undersampler.fit_resample(X, y)

print("Resampled class distribution:", Counter(y_resampled))

resampled_df = pd.DataFrame(X_resampled, columns=phishing_clean.drop('label', axis=1).columns)
resampled_df["label"] = y_resampled  # Add the label column

phishing_clean2 = resampled_df.copy()

Original class distribution: Counter({np.int64(1): 134850, np.int64(0): 100945})
Resampled class distribution: Counter({np.int64(0): 100945, np.int64(1): 100945})


In [ ]:
# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html
# https://scikit-learn.org/stable/modules/cross_validation.html

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score


X = phishing_clean2.drop('label', axis=1).values
y = phishing_clean2['label'].values

# Define models
models = {
    "Random Forest": RandomForestClassifier(),
    "XGBoost": XGBClassifier(),
    "Neural Network": MLPClassifier()
}

# Define scoring metrics
scoring = {
    "accuracy": make_scorer(accuracy_score),
    "precision": make_scorer(precision_score, average='macro'),
    "recall": make_scorer(recall_score, average='macro'),
    "f1": make_scorer(f1_score, average='macro')
}

# Perform cross-validation
kf = StratifiedKFold(n_splits=5)

for name, model in models.items():
    print(f"\n{name} Results:")
    for metric, scorer in scoring.items():
        scores = cross_val_score(model, X, y, cv=kf, scoring=scorer)
        print(f"{metric}: {scores.mean():.4f}")


Random Forest Results:
accuracy: 1.0000
precision: 1.0000
recall: 1.0000
f1: 1.0000

XGBoost Results:
accuracy: 1.0000
precision: 1.0000
recall: 1.0000
f1: 1.0000

Neural Network Results:
accuracy: 0.9992
precision: 0.9989
recall: 0.9989
